In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('./'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

./manifesto_sentences.csv
./meta_df.csv
./maindf.csv
./anno_df.csv
./notebooke111062ef6.ipynb
./preprocess_meta.joblib
./train.csv
./val.csv
./notebooke111062ef6 (copy).ipynb
./configuration_causal_interpretable.py
./modeling_causal_interpretable.py
./sentences.txt
./causal_nam_best_backup56c.pt
./Untitled.ipynb
./umap.pdf
./umap_six.pdf
./causal_nam_best_backup.pt
./change_by_year_woFiLM.pdf
./change_by_year.pdf
./causal_nam_best.pt
./.ipynb_checkpoints/notebooke111062ef6-checkpoint.ipynb
./.ipynb_checkpoints/configuration_causal_interpretable-checkpoint.py
./.ipynb_checkpoints/modeling_causal_interpretable-checkpoint.py
./.ipynb_checkpoints/sentences-checkpoint.txt
./.ipynb_checkpoints/Untitled-checkpoint.ipynb
./.ipynb_checkpoints/notebooke111062ef6 (copy)-checkpoint.ipynb
./outputs/6ctry_6macros_6sents_curves.csv
./outputs/6ctry_6macros_6sents_summary.csv
./outputs/change_by_year.pdf
./outputs/L1_compare.pdf
./outputs/L2_compare.pdf
./outputs/L3_compare.pdf
./outputs/S1_compare.pdf

In [2]:
#!pip install -q "protobuf==3.20.3"

In [3]:
# preprocess_data.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import joblib

DATA_PATH = Path("./manifesto_sentences.csv")

def load_and_prepare(min_count: int = 100):
    df = pd.read_csv(DATA_PATH, on_bad_lines='skip')  # 如果是逗号改成 sep=","

    # 1) 处理类别 label（category 可能是 408, 402 这种编码，先映射到 0..K-1）
    #cat_le = LabelEncoder()
    #df = df.dropna(subset=['category'])   
    #df["category_id"] = cat_le.fit_transform(df["category"])


    cat_le = LabelEncoder()
    
    df = df.dropna(subset=["category"]).copy()
    
    # 统一为字符串便于解析
    s = df["category"].astype(str).str.strip()
    
    # 1) 从 "per101" / "101" / "1011" 里提取数字部分；提取不到则为 NaN
    num = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
    
    # 2) 只保留三位主类别：101-706（会自动排除 1011 等四位码）
    df = df[num.between(101, 706)].copy()
    df["category_main"] = num[num.between(101, 706)].astype(int)
    
    # 3) 编码（如果你需要连续的 0..K-1）
    df["category_id"] = cat_le.fit_transform(df["category_main"])
    
    print("num_classes =", len(cat_le.classes_))
    print("min/max =", df["category_main"].min(), df["category_main"].max())


    # 2) 国家 / 年份 也编码（用于 embedding）
    country_le = LabelEncoder()
    df["country_id"] = country_le.fit_transform(df["country"])

    year_le = LabelEncoder()
    df["year_id"] = year_le.fit_transform(df["year"])

    # 3) 宏观变量列名
    macro_cols = [
        "battle_value",
        "gdp_value",
        "trade_value",
        "debt_value",
        "inflation_value",
        "unemployment_value"
    ]

    # 简单处理缺失值
    df[macro_cols] = df[macro_cols].fillna(0.0)

    # 可以做一个标准化（非必须，但对训练更稳定）
    macro_means = df[macro_cols].mean()
    macro_stds = df[macro_cols].std().replace(0, 1.0)
    df[macro_cols] = (df[macro_cols] - macro_means) / macro_stds
    df[macro_cols] = df[macro_cols].astype("float32")
    # 4) train/val 划分（按 doc_id 或随机都可，这里先简单随机）
    # 4) 过滤样本数太少的类别（关键修改）
    counts = df["category_id"].value_counts()
    valid_categories = counts[counts >= min_count].index
    df = df[df["category_id"].isin(valid_categories)].reset_index(drop=True)

    print(f"原始样本数: {len(counts)} 个类别，过滤后保留 {len(valid_categories)} 个类别")
    print(f"过滤后总样本数: {len(df)}")

    # 如果过滤后类别数还是很少，也可以进一步提高 min_count，比如 5
    # counts = df["category_id"].value_counts()
    # print(counts.head())

    # 5) train/val 划分（这时 stratify 就不会再报错了）
    train_df, val_df = train_test_split(
        df,
        test_size=0.1,
        random_state=42,
        stratify=df["category_id"]
    )

    # 6) 保存编码器和宏观变量标准化参数
    joblib.dump(
        {
            "cat_le": cat_le,
            "country_le": country_le,
            "year_le": year_le,
            "macro_means": macro_means,
            "macro_stds": macro_stds,
            "macro_cols": macro_cols
        },
        "preprocess_meta.joblib"
    )

    train_df.to_csv("train.csv", index=False);print('train:', train_df.shape)
    val_df.to_csv("val.csv", index=False);print('test:', val_df.shape)
    print("Saved train/val and preprocess_meta.")

if __name__ == "__main__":
    load_and_prepare()

num_classes = 56
min/max = 101 706
原始样本数: 56 个类别，过滤后保留 56 个类别
过滤后总样本数: 1901346
train: (1711211, 15)
test: (190135, 15)
Saved train/val and preprocess_meta.


In [4]:
# dataset.py
import torch
from torch.utils.data import Dataset
import pandas as pd
from transformers import AutoTokenizer
import joblib

class ManifestoSentenceDataset(Dataset):
    def __init__(self, csv_path, meta_path="preprocess_meta.joblib",
                 model_name="bert-base-multilingual-cased", max_len=128):
        self.df = pd.read_csv(csv_path)
        self.meta = joblib.load(meta_path)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.max_len = max_len

        self.macro_cols = self.meta["macro_cols"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        text = str(row["sentence"])
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )

        # 关键修改：明确转成 float32 的 numpy，再转 torch
        macro_vals = row[self.macro_cols].to_numpy(dtype="float32")
        macro = torch.from_numpy(macro_vals)  # (num_macro,)

        country_id = torch.tensor(row["country_id"], dtype=torch.long)
        year_id = torch.tensor(row["year_id"], dtype=torch.long)
        label = torch.tensor(row["category_id"], dtype=torch.long)

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "macro": macro,
            "country_id": country_id,
            "year_id": year_id,
            "labels": label
        }
        return item

/home/wang/anaconda3/envs/prollm/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# causal_nam_model.py
import torch
import torch.nn as nn
from transformers import AutoModel


class MacroSubNet(nn.Module):
    """Single macro variable: scalar -> per-class logit contribution (kept for interpretability)."""
    def __init__(self, num_classes: int, hidden_dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)  # (batch, num_classes)


class MacroTokenEncoder(nn.Module):
    """Encode each macro scalar into a token embedding."""
    def __init__(self, d_model: int, hidden: int = 64, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
        )

    def forward(self, x):  # x: (batch, 1)
        return self.net(x)  # (batch, d_model)


class CausalInterpretableModel(nn.Module):
    """
    Same inputs/outputs as your current model, but with stronger interaction between:
    text <-> (macro, country, year) via:
      (1) a structural-context Transformer over macro/country/year tokens
      (2) FiLM modulation of CLS embedding
      (3) an explicit interaction head (still decomposable)
    """
    def __init__(
        self,
        model_name="bert-base-multilingual-cased",
        num_classes=56,
        num_macro=6,
        num_countries=50,
        num_years=100,
        text_hidden_dim=256,
        macro_hidden_dim=16,
        country_emb_dim=16,
        year_emb_dim=16,
        ctx_dim=128,
        ctx_depth=2,
        ctx_heads=4,
        dropout=0.1,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.num_macro = num_macro

        # ----- Text encoder -----
        self.bert = AutoModel.from_pretrained(model_name)
        bert_dim = self.bert.config.hidden_size

        # Base text head (kept)
        self.text_proj = nn.Sequential(
            nn.Linear(bert_dim, text_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(text_hidden_dim, num_classes),
        )

        # ----- Additive macro heads (kept) -----
        self.macro_nets = nn.ModuleList([
            MacroSubNet(num_classes=num_classes, hidden_dim=macro_hidden_dim)
            for _ in range(num_macro)
        ])

        # ----- Country/Year embeddings (kept) -----
        self.country_emb = nn.Embedding(num_countries, country_emb_dim)
        self.country_head = nn.Linear(country_emb_dim, num_classes)

        self.year_emb = nn.Embedding(num_years, year_emb_dim)
        self.year_head = nn.Linear(year_emb_dim, num_classes)

        # ----- Structural context encoder (NEW) -----
        # macro scalars -> ctx tokens
        self.macro_token = MacroTokenEncoder(d_model=ctx_dim, hidden=max(64, ctx_dim // 2), dropout=dropout)

        # country/year -> ctx tokens
        self.country_to_ctx = nn.Sequential(
            nn.Linear(country_emb_dim, ctx_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.year_to_ctx = nn.Sequential(
            nn.Linear(year_emb_dim, ctx_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        enc_layer = nn.TransformerEncoderLayer(
            d_model=ctx_dim,
            nhead=ctx_heads,
            dim_feedforward=ctx_dim * 4,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.ctx_encoder = nn.TransformerEncoder(enc_layer, num_layers=ctx_depth)
        self.ctx_norm = nn.LayerNorm(ctx_dim)

        # ----- FiLM modulation of CLS by context (NEW) -----
        # gamma/beta in bert_dim space
        self.ctx_to_gamma_beta = nn.Sequential(
            nn.Linear(ctx_dim, bert_dim * 2),
            nn.Tanh(),  # keeps modulation bounded
        )

        # ----- Explicit interaction head (NEW) -----
        # produces per-class logit adjustments due to text-structure interaction
        self.ctx_to_bert = nn.Linear(ctx_dim, bert_dim)
        self.interaction_head = nn.Sequential(
            nn.Linear(bert_dim, text_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(text_hidden_dim, num_classes),
        )

    def _encode_context(self, macro, country_id, year_id):
        """
        macro: (batch, num_macro)
        returns:
          ctx: (batch, ctx_dim)
          country_vec/year_vec: embeddings for optional components output
        """
        country_vec = self.country_emb(country_id)  # (batch, country_emb_dim)
        year_vec = self.year_emb(year_id)          # (batch, year_emb_dim)

        # Build tokens: [macro_1 ... macro_M, country_token, year_token]
        macro_tokens = []
        for j in range(self.num_macro):
            xj = macro[:, j:j+1]               # (batch, 1)
            macro_tokens.append(self.macro_token(xj))  # (batch, ctx_dim)
        macro_tokens = torch.stack(macro_tokens, dim=1)  # (batch, M, ctx_dim)

        c_tok = self.country_to_ctx(country_vec).unsqueeze(1)  # (batch, 1, ctx_dim)
        y_tok = self.year_to_ctx(year_vec).unsqueeze(1)        # (batch, 1, ctx_dim)
        tokens = torch.cat([macro_tokens, c_tok, y_tok], dim=1)  # (batch, M+2, ctx_dim)

        # Let tokens interact
        tokens = self.ctx_encoder(tokens)  # (batch, M+2, ctx_dim)
        ctx = tokens.mean(dim=1)           # (batch, ctx_dim)
        ctx = self.ctx_norm(ctx)
        return ctx, country_vec, year_vec

    def forward(self, input_ids, attention_mask, macro, country_id, year_id, output_components=False):
        """
        Inputs/outputs kept the same:
          - If output_components=False: return logits
          - If True: return (logits, dict)
        """
        # ----- Context (macro+country+year) -----
        ctx, country_vec, year_vec = self._encode_context(macro, country_id, year_id)

        # ----- Text encoding -----
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = out.last_hidden_state[:, 0, :]  # (batch, bert_dim)

        # ----- FiLM modulation (text conditioned on context) -----
        gamma_beta = self.ctx_to_gamma_beta(ctx)          # (batch, 2*bert_dim)
        gamma, beta = gamma_beta.chunk(2, dim=-1)         # each (batch, bert_dim)
        cls_film = cls_emb * (1.0 + gamma) + beta         # (batch, bert_dim)

        # Text logits (now structurally conditioned)
        logits_text = self.text_proj(cls_film)            # (batch, num_classes)

        # ----- Additive macro contributions (kept) -----
        macro_contribs = []
        for j, net in enumerate(self.macro_nets):
            xj = macro[:, j:j+1]        # (batch, 1)
            macro_contribs.append(net(xj))
        logits_macro = torch.stack(macro_contribs, dim=0).sum(dim=0)  # (batch, num_classes)

        # ----- Country/year main effects (kept) -----
        logits_country = self.country_head(country_vec)   # (batch, num_classes)
        logits_year = self.year_head(year_vec)            # (batch, num_classes)

        # ----- Explicit interaction term (NEW, but still decomposable) -----
        ctx_in_bert = self.ctx_to_bert(ctx)               # (batch, bert_dim)
        interaction_feat = cls_emb * ctx_in_bert 
        logits_interaction = self.interaction_head(interaction_feat)  # (batch, num_classes)

        # ----- Total logits -----
        logits = logits_text + logits_interaction + logits_macro + logits_country + logits_year #

        if output_components:
            return logits, {
                "logits_text": logits_text,
                "logits_macro_each": macro_contribs,
                "logits_macro_sum": logits_macro,
                "logits_country": logits_country,
                "logits_year": logits_year,
                "logits_interaction": logits_interaction,  # extra, backward-compatible
            }
        return logits


In [6]:
class ManifestoDebugDataset(ManifestoSentenceDataset):
    """
    Debug dataset: only take first n samples from CSV.
    Usage:
        debug_ds = ManifestoDebugDataset("train.csv", n=64)
    """
    def __init__(self, csv_path, n=64,
                 meta_path="preprocess_meta.joblib",
                 model_name="bert-base-multilingual-cased",
                 max_len=128):

        super().__init__(
            csv_path=csv_path,
            meta_path=meta_path,
            model_name=model_name,
            max_len=max_len
        )

        # 截断 DataFrame
        self.df = self.df.iloc[:n].reset_index(drop=True)
        print(f"[DEBUG DATA] Loaded {len(self.df)} samples from {csv_path}")

def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

In [7]:
# train_causal_nam.py
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from tqdm import tqdm
import pandas as pd
import joblib
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    classification_report,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

meta = joblib.load("preprocess_meta.joblib")
num_classes = len(meta["cat_le"].classes_)
num_countries = len(meta["country_le"].classes_)
num_years = len(meta["year_le"].classes_)
num_macro = len(meta["macro_cols"])

train_ds = ManifestoSentenceDataset("train.csv")
val_ds = ManifestoSentenceDataset("val.csv")
#train_ds = ManifestoDebugDataset("train.csv", n=2560)
#val_ds = ManifestoDebugDataset("val.csv", n=1280)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

model = CausalInterpretableModel(
    model_name="bert-base-multilingual-cased",
    num_classes=num_classes,
    num_macro=num_macro,
    num_countries=num_countries,
    num_years=num_years
).to(device)



optimizer = AdamW(model.parameters(), lr=2e-5)
criterion = CrossEntropyLoss()

best_val_loss = float("inf")

for epoch in range(5):  # 先跑几轮看看
    # ======================
    #  Train
    # ======================
    model.train()
    total_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - Train"):
        batch = {k: v.to(device) for k, v in batch.items()}

        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            macro=batch["macro"],
            country_id=batch["country_id"],
            year_id=batch["year_id"],
        )

        loss = criterion(logits, batch["labels"])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    # ======================
    #  Validation + 多指标
    # ======================
    model.eval()
    val_loss = 0.0

    all_labels = []
    all_preds = []
    top3_correct = 0
    top5_correct = 0
    total = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - Val"):
            batch = {k: v.to(device) for k, v in batch.items()}

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                macro=batch["macro"],
                country_id=batch["country_id"],
                year_id=batch["year_id"],
            )

            loss = criterion(logits, batch["labels"])
            val_loss += loss.item()

            # 基本预测
            preds = logits.argmax(dim=1)

            all_labels.append(batch["labels"].cpu().numpy())
            all_preds.append(preds.cpu().numpy())

            # Top-k 统计
            total += batch["labels"].size(0)
            # top-3
            top3 = torch.topk(logits, k=min(3, logits.size(1)), dim=1).indices  # (B, 3)
            top3_correct += (
                (top3 == batch["labels"].unsqueeze(1)).any(dim=1).sum().item()
            )
            # top-5（如果类别数 >= 5）
            if logits.size(1) >= 5:
                top5 = torch.topk(logits, k=5, dim=1).indices
                top5_correct += (
                    (top5 == batch["labels"].unsqueeze(1)).any(dim=1).sum().item()
                )

    avg_val_loss = val_loss / len(val_loader)

    # 拼接所有 batch 的 label / pred
    all_labels = np.concatenate(all_labels, axis=0)
    all_preds = np.concatenate(all_preds, axis=0)

    # ========== 计算多种指标 ==========
    val_acc = accuracy_score(all_labels, all_preds)
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    macro_precision, macro_recall, _, _ = precision_recall_fscore_support(
        all_labels, all_preds, average="macro", zero_division=0
    )
    top3_acc = top3_correct / total
    top5_acc = top5_correct / total if logits.size(1) >= 5 else None

    print("=" * 80)
    print(f"Epoch {epoch+1}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val   Loss: {avg_val_loss:.4f}")
    print(f"Val   Acc : {val_acc:.4f}")
    print(f"Macro F1  : {macro_f1:.4f}")
    print(f"Macro Prec: {macro_precision:.4f}")
    print(f"Macro Rec : {macro_recall:.4f}")
    print(f"Top-3 Acc : {top3_acc:.4f}")
    if top5_acc is not None:
        print(f"Top-5 Acc : {top5_acc:.4f}")
    print("-" * 80)
    # 如果你需要详细每类的指标，也可以解开这一行：
    #print(classification_report(all_labels, all_preds, digits=3))
    print("=" * 80)

    # 模型保存逻辑还是用 val_loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), "causal_nam_best.pt")
        print("Saved best model.")



Epoch 1 - Val: 100%|████████████████████████| 1486/1486 [07:52<00:00,  3.14it/s]


Epoch 1
Train Loss: 1.9452
Val   Loss: 1.6366
Val   Acc : 0.5422
Macro F1  : 0.3895
Macro Prec: 0.4724
Macro Rec : 0.3714
Top-3 Acc : 0.7737
Top-5 Acc : 0.8492
--------------------------------------------------------------------------------
Saved best model.


Epoch 2 - Val: 100%|████████████████████████| 1486/1486 [07:53<00:00,  3.14it/s]


Epoch 2
Train Loss: 1.5926
Val   Loss: 1.5332
Val   Acc : 0.5639
Macro F1  : 0.4379
Macro Prec: 0.5055
Macro Rec : 0.4137
Top-3 Acc : 0.7965
Top-5 Acc : 0.8688
--------------------------------------------------------------------------------
Saved best model.


Epoch 3 - Val: 100%|████████████████████████| 1486/1486 [07:54<00:00,  3.13it/s]


Epoch 3
Train Loss: 1.4404
Val   Loss: 1.4801
Val   Acc : 0.5753
Macro F1  : 0.4630
Macro Prec: 0.5162
Macro Rec : 0.4416
Top-3 Acc : 0.8073
Top-5 Acc : 0.8770
--------------------------------------------------------------------------------
Saved best model.


Epoch 4 - Val: 100%|████████████████████████| 1486/1486 [07:53<00:00,  3.14it/s]


Epoch 4
Train Loss: 1.3229
Val   Loss: 1.4616
Val   Acc : 0.5839
Macro F1  : 0.4753
Macro Prec: 0.5201
Macro Rec : 0.4583
Top-3 Acc : 0.8115
Top-5 Acc : 0.8802
--------------------------------------------------------------------------------
Saved best model.


Epoch 5 - Val: 100%|████████████████████████| 1486/1486 [07:53<00:00,  3.14it/s]


Epoch 5
Train Loss: 1.2196
Val   Loss: 1.4562
Val   Acc : 0.5885
Macro F1  : 0.4880
Macro Prec: 0.5321
Macro Rec : 0.4641
Top-3 Acc : 0.8138
Top-5 Acc : 0.8822
--------------------------------------------------------------------------------
Saved best model.


In [ ]:
import torch
import numpy as np
import joblib
from transformers import AutoTokenizer


meta = joblib.load("preprocess_meta.joblib")
macro_cols = meta["macro_cols"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 加载模型
model = CausalInterpretableModel(
    model_name="bert-base-multilingual-cased",
    num_classes=len(meta["cat_le"].classes_),
    num_macro=len(macro_cols),
    num_countries=len(meta["country_le"].classes_),
    num_years=len(meta["year_le"].classes_)
).to(device)
model.load_state_dict(torch.load("./w/causal_bert56c.pt", map_location=device))
model.eval()

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

sentence = "景気の減速と失業率の上昇を受け、政府は雇用維持と再就職支援を一体で進め、訓練・配置転換・中小企業支援を通じて長期失業の拡大を防ぐ方針である。"
country_str = "Japan"
year_val = 2021

country_id = meta["country_le"].transform([country_str])[0]
year_id = meta["year_le"].transform([year_val])[0]

base_macro = np.zeros(len(macro_cols), dtype=np.float32)  # 先用 0 向量（标准化之后的 0）
# 或者用该年该国的平均值

# 在失业率维度做一个 scan
unemp_idx = macro_cols.index("unemployment_value")
grid = np.linspace(-2, 2, 21)  # 从 -2σ 到 +2σ

probs_list = []

for u in grid:
    macro_vec = base_macro.copy()
    macro_vec[unemp_idx] = u

    enc = tokenizer(
        sentence,
        truncation=True,
        max_length=128,
        padding="max_length",
        return_tensors="pt"
    )

    with torch.no_grad():
        logits, comps = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
            macro=torch.tensor(macro_vec, dtype=torch.float32, device=device).unsqueeze(0),
            country_id=torch.tensor([country_id], dtype=torch.long, device=device),
            year_id=torch.tensor([year_id], dtype=torch.long, device=device),
            output_components=True
        )
        probs = logits.softmax(dim=-1).cpu().numpy()[0]
        probs_list.append(probs)

# probs_list 就是不同失业率下，policy 类别分布的变化
# 你可以画出某几个重点 policy 类别的 prob vs unemployment_value 曲线

In [ ]:
import matplotlib.pyplot as plt

probs_array = np.stack(probs_list, axis=0)  # (len(grid), num_classes)

# ======================
# 3. 选几个最重要的类别画图
# ======================
# 在失业率 = 0 的点上取概率最高的前 K 个类别
zero_idx = np.argmin(np.abs(grid - 0.0))
base_probs = probs_array[zero_idx]  # (num_classes,)
K = 5
topk_idx = np.argsort(base_probs)[-K:][::-1]  # 概率从大到小的前 K 类

cat_le = meta["cat_le"]
topk_labels = [cat_le.inverse_transform([i])[0] for i in topk_idx]

plt.figure(figsize=(8, 5))

for cls_idx, lbl in zip(topk_idx, topk_labels):
    plt.plot(
        grid,
        probs_array[:, cls_idx],
        marker="o",
        linewidth=1.5,
        label=f"category={lbl}"
    )

plt.xlabel("Standardized unemployment_value (σ)")
plt.ylabel("Predicted probability")
plt.title("Effect of unemployment on policy category probabilities")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import torch
import numpy as np
import joblib
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

meta = joblib.load("preprocess_meta.joblib")
macro_cols = meta["macro_cols"]
cat_le = meta["cat_le"]
country_le = meta["country_le"]
year_le = meta["year_le"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 你已加载好的模型（取消注释并确保 model 在作用域里）
# model = CausalInterpretableModel(...)
# model.load_state_dict(torch.load("models/causal_nam_best.pt", map_location=device))
# model.eval()

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

sentence = "ワクチン接種の強化が急務だ"
country_str = "Japan"

country_id = int(country_le.transform([country_str])[0])

# 固定 macro（标准化空间）；你也可以改成该国该年的均值
base_macro = np.zeros(len(macro_cols), dtype=np.float32)

# ==========
# 1) 扫描年份（x-axis = year）
# ==========
# year_le.classes_ 里存的是训练时出现过的年份（可能是 int 或 str）
years_all = year_le.classes_.tolist()
years_all = [int(y) for y in years_all]
years_all = sorted(years_all)

# 可选：限制年份范围（按需改）
start_year, end_year = 1990, 2023
years = [y for y in years_all if start_year <= y <= end_year]
years = [2000, 2003,2005,2009, 2012, 2014, 2017, 2021]
if len(years) == 0:
    raise ValueError(f"No years in [{start_year}, {end_year}] found in year_le.classes_.")

probs_list = []

for y in years:
    year_id = int(year_le.transform([y])[0])

    enc = tokenizer(
            sentence,
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
    )

    with torch.no_grad():
        logits, comps = model(
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
                macro=torch.tensor(base_macro, dtype=torch.float32, device=device).unsqueeze(0),
                country_id=torch.tensor([country_id], dtype=torch.long, device=device),
                year_id=torch.tensor([year_id], dtype=torch.long, device=device),
                output_components=True
        )
        probs = logits.softmax(dim=-1).cpu().numpy()[0]
        probs_list.append(probs)

probs_array = np.stack(probs_list, axis=0) # (len(years), num_classes)

# ==========
# 2) 选 topK 类别（在参考年份上取）
# ==========
ref_year = 2006
# 找到最接近 ref_year 的年份位置
ref_idx = int(np.argmin(np.abs(np.array(years) - ref_year)))
ref_probs = probs_array[ref_idx]

K = 5
topk_idx = np.argsort(ref_probs)[-K:][::-1]
topk_labels = [cat_le.inverse_transform([i])[0] for i in topk_idx]

# ==========
# 3) 画图：prob vs year
# ==========
plt.figure(figsize=(9, 5))

for cls_idx, lbl in zip(topk_idx, topk_labels):
    plt.plot(
            years,
            probs_array[:, cls_idx],
            marker="o",
            linewidth=1.5,
            label=f"category={lbl}"
    )

plt.xlabel("Year")
plt.ylabel("Predicted probability")
plt.title(f"Effect of year on policy category probabilities (country={country_str})")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig('change_by_year.pdf')
plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from difflib import get_close_matches
from transformers import AutoTokenizer


def _default_texts_3short_3long_6countries():
    """
    6 countries × (3 short + 3 long) sentences.
    (No reuse of your earlier sentences.)
    Countries: US, UK, Japan, South Korea, Germany, Finland
    """
    return {
        "United States": [
            {"sentence_id": "US_S1", "length": "short", "text": "Jobs are falling; households need immediate relief."},
            {"sentence_id": "US_S2", "length": "short", "text": "We will expand training and speed up hiring."},
            {"sentence_id": "US_S3", "length": "short", "text": "Inflation is squeezing families; prices must stabilize."},
            {"sentence_id": "US_L1", "length": "long", "text": "As unemployment rises and growth slows, the administration proposes a coordinated package combining targeted hiring subsidies, expanded apprenticeships, and temporary support for low-income households to prevent long-term labour-market scarring."},
            {"sentence_id": "US_L2", "length": "long", "text": "With debt-service costs increasing and inflation pressures persisting, we will reprioritize spending toward productivity-enhancing infrastructure while protecting core social programs, ensuring fiscal sustainability without undermining recovery."},
            {"sentence_id": "US_L3", "length": "long", "text": "To reduce vulnerability to external shocks, we will diversify supply chains and adjust trade policy while investing in innovation and energy security so that macroeconomic volatility does not translate into political instability."},
        ],
        "United Kingdom": [
            {"sentence_id": "UK_S1", "length": "short", "text": "We will defend jobs and protect wages."},
            {"sentence_id": "UK_S2", "length": "short", "text": "Public services must remain reliable in hard times."},
            {"sentence_id": "UK_S3", "length": "short", "text": "Trade must work for local communities."},
            {"sentence_id": "UK_L1", "length": "long", "text": "In response to weakening growth and rising unemployment, the government will combine labour-market activation with time-limited income support, aiming to raise participation while preventing long-run exclusion."},
            {"sentence_id": "UK_L2", "length": "long", "text": "With inflation elevated and fiscal space constrained, we will recalibrate tax and spending priorities toward cost-effective programmes, keeping debt dynamics credible while shielding vulnerable households from price shocks."},
            {"sentence_id": "UK_L3", "length": "long", "text": "We will pursue openness in trade where it raises productivity, but strengthen domestic resilience through skills, industrial strategy, and regional investment so that external exposure does not widen inequalities."},
        ],
        "Japan": [
            {"sentence_id": "JP_S1", "length": "short", "text": "雇用を守り、家計の不安を減らす。"},
            {"sentence_id": "JP_S2", "length": "short", "text": "物価上昇への対策を急ぐ。"},
            {"sentence_id": "JP_S3", "length": "short", "text": "地域の産業基盤を強化する。"},
            {"sentence_id": "JP_L1", "length": "long", "text": "景気の減速と失業率の上昇を受け、政府は雇用維持と再就職支援を一体で進め、訓練・配置転換・中小企業支援を通じて長期失業の拡大を防ぐ方針である。"},
            {"sentence_id": "JP_L2", "length": "long", "text": "財政余力が限られる中でも、国債費の増加とインフレの影響を踏まえ、給付と投資の優先順位を見直し、持続可能性を確保しつつ生活必需品の価格高騰に対応する。"},
            {"sentence_id": "JP_L3", "length": "long", "text": "貿易環境の変化や外的ショックに備え、供給網の強靭化と技術投資を進めることで、生産性の向上と賃金の底上げを同時に実現し、社会不安の拡大を抑える。"},
        ],
        "South Korea": [
            {"sentence_id": "KR_S1", "length": "short", "text": "고용을 지키고 청년의 기회를 넓히겠다."},
            {"sentence_id": "KR_S2", "length": "short", "text": "물가 안정이 최우선이다."},
            {"sentence_id": "KR_S3", "length": "short", "text": "무역의 성과가 모두에게 돌아가야 한다."},
            {"sentence_id": "KR_L1", "length": "long", "text": "경기 둔화와 실업률 상승에 대응해 정부는 채용 인센티브와 직업훈련을 결합하고, 취약계층의 소득 공백을 완화하여 장기 실업으로의 이행을 억제하겠다."},
            {"sentence_id": "KR_L2", "length": "long", "text": "부채 부담과 인플레이션 압력이 동시에 커지는 상황에서, 재정의 지속가능성을 훼손하지 않도록 지출 구조를 조정하되 필수 복지와 생산성 투자에 우선순위를 두겠다."},
            {"sentence_id": "KR_L3", "length": "long", "text": "대외 환경 변화에 따른 무역 노출을 관리하면서 공급망 안정과 산업 혁신을 추진해, 외부 충격이 불평등과 사회 갈등으로 전이되지 않도록 제도적 완충장치를 강화하겠다."},
        ],
        "Germany": [
            {"sentence_id": "DE_S1", "length": "short", "text": "Wir sichern Arbeit und stärken faire Löhne."},
            {"sentence_id": "DE_S2", "length": "short", "text": "Inflation belastet Familien; wir handeln sofort."},
            {"sentence_id": "DE_S3", "length": "short", "text": "Der Staat muss investieren, ohne die Zukunft zu verspielen."},
            {"sentence_id": "DE_L1", "length": "long", "text": "Angesichts schwächerer Konjunktur und steigender Arbeitslosigkeit setzen wir auf Qualifizierung, gezielte Entlastungen und Investitionen, damit Unternehmen Beschäftigung halten und Menschen schnell in neue Arbeit finden."},
            {"sentence_id": "DE_L2", "length": "long", "text": "Wenn Zinskosten und Schuldenquoten steigen, priorisieren wir wirksame Ausgaben und stärken die Produktivität, um fiskalische Stabilität zu sichern und zugleich sozialen Zusammenhalt zu bewahren."},
            {"sentence_id": "DE_L3", "length": "long", "text": "Wir verbinden offene Märkte mit einer resilienten Industriepolitik, stärken Lieferketten und Energieversorgung und sorgen dafür, dass Handelsabhängigkeit nicht zu ökonomischer Unsicherheit und politischer Polarisierung führt."},
        ],
        "Finland": [
            {"sentence_id": "FI_S1", "length": "short", "text": "Työllisyys on turvattava ja eriarvoisuus pienennettävä."},
            {"sentence_id": "FI_S2", "length": "short", "text": "Hintojen nousu ei saa murentaa arkea."},
            {"sentence_id": "FI_S3", "length": "short", "text": "Avoin kauppa tarvitsee reilut pelisäännöt."},
            {"sentence_id": "FI_L1", "length": "long", "text": "Talouden hidastuessa ja työttömyyden kasvaessa toteutamme koulutuksen ja työllisyyspalveluiden uudistuksen sekä kohdennetun tuen, jotta pitkäaikaistyöttömyys ei pääse juurtumaan."},
            {"sentence_id": "FI_L2", "length": "long", "text": "Kun inflaatio ja julkisen talouden paineet kasvavat, kohdistamme menot vaikuttavasti ja vahvistamme investointeja, jotta velkakestävyys paranee ilman, että peruspalvelut heikkenevät."},
            {"sentence_id": "FI_L3", "length": "long", "text": "Hallitsemme ulkoista haavoittuvuutta vahvistamalla huoltovarmuutta ja innovaatioita sekä huolehtimalla siitä, ettei kaupan avautuminen johda alueelliseen eriytymiseen tai sosiaalisiin jännitteisiin."},
        ],
    }


def _resolve_label(label, classes, alias=None, fuzzy_cutoff=0.80):
    if label in classes:
        return label
    if alias and label in alias:
        for cand in alias[label]:
            if cand in classes:
                return cand
    if alias:
        for k, cands in alias.items():
            if label in cands and k in classes:
                return k
    m = get_close_matches(label, list(classes), n=1, cutoff=fuzzy_cutoff)
    if m:
        return m[0]
    raise ValueError(f"Unseen label: {label}")


@torch.no_grad()
def build_structural_sensitivity_csv_v3(
    model,
    meta,
    year_val: int,
    output_dir: str,
    output_prefix: str = "struct_sensitivity",
    countries=("United States", "United Kingdom", "Japan", "South Korea", "Germany", "Finland"),
    texts_by_country=None,
    tokenizer_name: str = "bert-base-multilingual-cased",
    macro_grid_spec=None,
    base_macro_mode: str = "zeros",   # "zeros" | "custom"
    base_macro_custom=None,
    store_mode: str = "topk",         # "topk" | "all" | "selected"
    topk: int = 10,
    selected_categories=None,
    scenarios_df: pd.DataFrame = None,  # optional: adjust all variables simultaneously
    device=None,
    max_length: int = 128,
    fuzzy_cutoff: float = 0.80,
):
    """
    不打印；只写 CSV。
    6 countries × (3 short + 3 long) sentences，且支持：
      - one-at-a-time 扫描每个宏变量
      - 或传 scenarios_df 同时调整所有宏变量组合
    输出：
      - {prefix}_curves.csv
      - {prefix}_summary.csv （仅 one-at-a-time 模式）
    """
    os.makedirs(output_dir, exist_ok=True)

    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    macro_cols = meta["macro_cols"]
    cat_le = meta["cat_le"]
    country_le = meta["country_le"]
    year_le = meta["year_le"]

    # 扩展 alias（德/芬一般不会冲突；韩/美/英常见别名仍保留）
    country_alias = {
        "South Korea": ["Korea, Rep.", "Korea (South)", "Republic of Korea", "Korea"],
        "Republic of Korea": ["South Korea", "Korea (South)", "Korea"],
        "UK": ["United Kingdom", "Britain", "Great Britain"],
        "USA": ["United States", "United States of America", "US"],
        "Deutschland": ["Germany"],
        "Suomi": ["Finland"],
    }

    if texts_by_country is None:
        texts_by_country = _default_texts_3short_3long_6countries()

    if macro_grid_spec is None:
        macro_grid_spec = {m: (-2.0, 2.0, 21) for m in macro_cols}

    def make_grid(spec):
        if isinstance(spec, tuple) and len(spec) == 3:
            a, b, n = spec
            return np.linspace(a, b, int(n)).astype(np.float32)
        arr = np.asarray(spec, dtype=np.float32)
        if arr.ndim != 1:
            raise ValueError("macro_grid_spec 每个变量必须是一维数组或(min,max,n)")
        return arr

    macro_grids = {m: make_grid(macro_grid_spec[m]) for m in macro_cols}

    year_id = int(year_le.transform([year_val])[0])

    def encode_country(country_str):
        resolved = _resolve_label(country_str, country_le.classes_, alias=country_alias, fuzzy_cutoff=fuzzy_cutoff)
        cid = int(country_le.transform([resolved])[0])
        return cid, resolved

    def get_base_macro():
        if base_macro_mode == "zeros":
            return np.zeros(len(macro_cols), dtype=np.float32)
        if base_macro_mode == "custom":
            if base_macro_custom is None:
                raise ValueError("base_macro_mode='custom' 需要 base_macro_custom")
            v = np.asarray(base_macro_custom, dtype=np.float32)
            if v.shape != (len(macro_cols),):
                raise ValueError(f"base_macro_custom 形状必须是 ({len(macro_cols)},)")
            return v.copy()
        raise ValueError("base_macro_mode 只能是 'zeros' 或 'custom'")

    def forward(sentence, macro_vec, country_id):
        enc = tokenizer(
            sentence, truncation=True, max_length=max_length, padding="max_length", return_tensors="pt"
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
            macro=torch.tensor(macro_vec, dtype=torch.float32, device=device).unsqueeze(0),
            country_id=torch.tensor([country_id], dtype=torch.long, device=device),
            year_id=torch.tensor([year_id], dtype=torch.long, device=device),
            output_components=False,
        )
        return logits.softmax(dim=-1).cpu().numpy()[0]

    # category selection
    all_cat_labels = list(cat_le.classes_)
    if store_mode == "selected":
        if not selected_categories:
            raise ValueError("store_mode='selected' 需要 selected_categories")
        selected_idx = []
        for lab in selected_categories:
            lab_res = _resolve_label(lab, all_cat_labels, alias=None, fuzzy_cutoff=0.999)
            selected_idx.append(int(np.where(cat_le.classes_ == lab_res)[0][0]))
        selected_idx = np.array(selected_idx, dtype=int)
    else:
        selected_idx = None

    curves = []
    summary = []

    base_macro = get_base_macro()

    for country in countries:
        if country not in texts_by_country:
            raise ValueError(f"texts_by_country 缺少 {country}")
        country_id, country_resolved = encode_country(country)

        for sent_obj in texts_by_country[country]:
            sid = sent_obj["sentence_id"]
            slen = sent_obj["length"]
            text = sent_obj["text"]

            base_probs = forward(text, base_macro, country_id)

            if store_mode == "all":
                idx_to_store = np.arange(len(base_probs), dtype=int)
            elif store_mode == "selected":
                idx_to_store = selected_idx
            else:
                idx_to_store = np.argsort(base_probs)[-topk:][::-1]

            labels_to_store = [cat_le.inverse_transform([i])[0] for i in idx_to_store]

            # ===== scenarios mode: adjust all macros simultaneously =====
            if scenarios_df is not None:
                df = scenarios_df.copy()
                miss = [m for m in macro_cols if m not in df.columns]
                if miss:
                    raise ValueError(f"scenarios_df 缺少列: {miss}")
                if "scenario_id" not in df.columns:
                    df["scenario_id"] = np.arange(len(df), dtype=int)

                for _, r in df.iterrows():
                    macro_vec = r[macro_cols].astype(np.float32).to_numpy()
                    probs = forward(text, macro_vec, country_id)
                    for cls_i, cls_lbl in zip(idx_to_store, labels_to_store):
                        curves.append({
                            "mode": "scenario",
                            "scenario_id": int(r["scenario_id"]),
                            "country_input": country,
                            "country_resolved": country_resolved,
                            "year": year_val,
                            "sentence_id": sid,
                            "sentence_length": slen,
                            "sentence": text,
                            "macro": "__all__",
                            "macro_value_std": np.nan,
                            "category": cls_lbl,
                            "class_index": int(cls_i),
                            "prob": float(probs[cls_i]),
                            "baseline_prob": float(base_probs[cls_i]),
                            "delta_vs_baseline": float(probs[cls_i] - base_probs[cls_i]),
                        })
                continue

            # ===== one-at-a-time mode =====
            for m_i, macro_name in enumerate(macro_cols):
                grid_vals = macro_grids[macro_name]
                y = np.zeros((len(grid_vals), len(idx_to_store)), dtype=np.float32)

                for gi, v in enumerate(grid_vals):
                    macro_vec = base_macro.copy()
                    macro_vec[m_i] = v
                    probs = forward(text, macro_vec, country_id)

                    y[gi, :] = probs[idx_to_store]

                    for cls_i, cls_lbl in zip(idx_to_store, labels_to_store):
                        curves.append({
                            "mode": "one_at_a_time",
                            "scenario_id": np.nan,
                            "country_input": country,
                            "country_resolved": country_resolved,
                            "year": year_val,
                            "sentence_id": sid,
                            "sentence_length": slen,
                            "sentence": text,
                            "macro": macro_name,
                            "macro_value_std": float(v),
                            "category": cls_lbl,
                            "class_index": int(cls_i),
                            "prob": float(probs[cls_i]),
                            "baseline_prob": float(base_probs[cls_i]),
                            "delta_vs_baseline": float(probs[cls_i] - base_probs[cls_i]),
                        })

                lo, hi = 0, len(grid_vals) - 1
                delta = y[hi, :] - y[lo, :]

                zero_idx = int(np.argmin(np.abs(grid_vals - 0.0)))
                if 0 < zero_idx < len(grid_vals) - 1:
                    slope0 = (y[zero_idx + 1, :] - y[zero_idx - 1, :]) / (
                        float(grid_vals[zero_idx + 1] - grid_vals[zero_idx - 1])
                    )
                else:
                    slope0 = np.full((len(idx_to_store),), np.nan, dtype=np.float32)

                max_min = y.max(axis=0) - y.min(axis=0)
                auc = np.trapz(y, grid_vals, axis=0)

                for j, (cls_i, cls_lbl) in enumerate(zip(idx_to_store, labels_to_store)):
                    summary.append({
                        "country_input": country,
                        "country_resolved": country_resolved,
                        "year": year_val,
                        "sentence_id": sid,
                        "sentence_length": slen,
                        "macro": macro_name,
                        "category": cls_lbl,
                        "class_index": int(cls_i),
                        "baseline_prob": float(base_probs[cls_i]),
                        "delta(+hi-(-lo))": float(delta[j]),
                        "slope@0": float(slope0[j]) if np.isfinite(slope0[j]) else np.nan,
                        "max-min": float(max_min[j]),
                        "auc": float(auc[j]),
                        "grid_min": float(grid_vals[0]),
                        "grid_max": float(grid_vals[-1]),
                        "grid_n": int(len(grid_vals)),
                    })

    curves_df = pd.DataFrame(curves)
    summary_df = pd.DataFrame(summary)

    curves_path = os.path.join(output_dir, f"{output_prefix}_curves.csv")
    summary_path = os.path.join(output_dir, f"{output_prefix}_summary.csv")

    curves_df.to_csv(curves_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    return curves_path, summary_path



In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def plot_nhb_comparison_from_csv(
    curves_csv: str,
    output_png: str,
    summary_csv: str = None,
    # sentence selection
    sentence_length_filter: str = "long",          # "short" | "long" | None
    sentence_id_filter: list = None,              # e.g., ["US_L1","UK_L1","JP_L1","KR_L1"] if you want fixed IDs
    choose_sentence_per_country: str = "max_baseline_prob",  # "first" | "max_baseline_prob"
    # category selection (what line to plot)
    plot_mode: str = "top1_by_baseline",          # "top1_by_baseline" | "fixed_category"
    fixed_category: str = None,                   # required if plot_mode="fixed_category"
    # macro selection/order
    macro_cols_order: list = None,                # explicit order of 6 macros
    # smoothing/interpolation for publication-grade curves
    interpolate_points: int = 200,                # >0 to interpolate curves
    # uncertainty band (requires multiple sentences per country OR summary_csv with per-point dispersion)
    add_band: bool = False,                       # if True, uses across-sentence variability as band
    band_stat: str = "std",                       # "std" | "sem" | "iqr"
    # figure layout
    nrows: int = 2,
    ncols: int = 3,
    figsize=(8.5, 5.6),                           # close to single-column Nature-ish aspect when saved
    dpi: int = 300,
    # aesthetics (Nature Human Behaviour-like)
    font_family: str = "Arial",
    base_fontsize: int = 9,
    line_width: float = 1.6,
    marker_size: float = 3.0,
    show_markers: bool = True,
    legend_ncol: int = 2,
):
    """
    Publication-style 2x3 panel figure (Nature Human Behaviour-like):
      - clean spines, minimal grid, consistent typography
      - optional interpolation for smooth curves
      - optional uncertainty bands across sentences (if multiple sentences per country remain after filtering)

    Input CSV must be the curves file produced by build_structural_sensitivity_csv (mode == one_at_a_time).
    Saves a single PNG and returns output_png.
    """
    df = pd.read_csv(curves_csv)
    df = df[df["mode"] == "one_at_a_time"].copy()

    # filter sentence length
    if sentence_length_filter is not None:
        df = df[df["sentence_length"] == sentence_length_filter].copy()

    # optional explicit sentence_id filter
    if sentence_id_filter is not None:
        df = df[df["sentence_id"].isin(sentence_id_filter)].copy()

    # macro order
    if macro_cols_order is None:
        macro_cols_order = list(df["macro"].dropna().unique())
    macro_cols_order = [m for m in macro_cols_order if m != "__all__"]
    macro_cols_order = macro_cols_order[: (nrows * ncols)]

    # choose one sentence per country (if not explicitly filtered)
    if sentence_id_filter is None:
        chosen = []
        for c in df["country_input"].unique():
            sub = df[df["country_input"] == c].copy()
            if sub.empty:
                continue
            if choose_sentence_per_country == "max_baseline_prob":
                sid = (
                    sub.groupby("sentence_id")["baseline_prob"]
                    .max()
                    .sort_values(ascending=False)
                    .index[0]
                )
            else:
                sid = sub["sentence_id"].iloc[0]
            chosen.append((c, sid))
        keep = pd.DataFrame(chosen, columns=["country_input", "sentence_id"])
        df = df.merge(keep, on=["country_input", "sentence_id"], how="inner")

    # category selection per panel
    if plot_mode == "fixed_category":
        if fixed_category is None:
            raise ValueError("plot_mode='fixed_category' requires fixed_category")
        df_plot = df[df["category"] == fixed_category].copy()
        label_map = {c: f"{c} ({fixed_category})" for c in df_plot["country_input"].unique()}
    else:
        # pick top-1 category by baseline for each (country, macro)
        pick = (
            df.groupby(["country_input", "macro", "category"], as_index=False)["baseline_prob"]
            .max()
            .sort_values(["country_input", "macro", "baseline_prob"], ascending=[True, True, False])
            .groupby(["country_input", "macro"], as_index=False)
            .head(1)
        )
        df_plot = df.merge(
            pick[["country_input", "macro", "category"]],
            on=["country_input", "macro", "category"],
            how="inner"
        )
        # for legend, include chosen category per country using first macro occurrence
        tmp = (
            df_plot.groupby(["country_input", "category"], as_index=False)["baseline_prob"]
            .max()
            .sort_values(["country_input", "baseline_prob"], ascending=[True, False])
            .groupby("country_input", as_index=False)
            .head(1)
        )
        cat_by_country = dict(zip(tmp["country_input"], tmp["category"]))
        label_map = {c: f"{c} ({cat_by_country.get(c,'')})" for c in df_plot["country_input"].unique()}

    # ---------- Style (Nature-like) ----------
    plt.rcParams.update({
        #"font.family": font_family,
        "font.size": base_fontsize,
        "axes.titlesize": base_fontsize + 1,
        "axes.labelsize": base_fontsize,
        "legend.fontsize": base_fontsize - 1,
        "xtick.labelsize": base_fontsize - 1,
        "ytick.labelsize": base_fontsize - 1,
        "axes.linewidth": 0.8,
        "figure.dpi": dpi,
        "savefig.dpi": dpi,
    })

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=False)
    axes = np.array(axes).reshape(-1)

    # consistent x-range per macro from data
    x_min = df_plot["macro_value_std"].min()
    x_max = df_plot["macro_value_std"].max()

    def _interp(x, y, n=200):
        # deterministic linear interpolation (no SciPy dependency)
        xi = np.linspace(float(np.min(x)), float(np.max(x)), int(n))
        yi = np.interp(xi, x, y)
        return xi, yi

    # uncertainty band helper
    def _band_stats(y_stack):
        # y_stack: (n_series, n_grid) aligned on same x
        if band_stat == "sem":
            mu = np.nanmean(y_stack, axis=0)
            sd = np.nanstd(y_stack, axis=0, ddof=1)
            n = np.sum(np.isfinite(y_stack), axis=0)
            se = np.divide(sd, np.sqrt(np.maximum(n, 1)), out=np.zeros_like(sd), where=n > 0)
            return mu, mu - se, mu + se
        if band_stat == "iqr":
            mu = np.nanmedian(y_stack, axis=0)
            q1 = np.nanpercentile(y_stack, 25, axis=0)
            q3 = np.nanpercentile(y_stack, 75, axis=0)
            return mu, q1, q3
        # default std
        mu = np.nanmean(y_stack, axis=0)
        sd = np.nanstd(y_stack, axis=0, ddof=1)
        return mu, mu - sd, mu + sd

    for i, macro_name in enumerate(macro_cols_order):
        ax = axes[i]
        subm = df_plot[df_plot["macro"] == macro_name].copy()

        # panel title: macro name (you can rename outside if needed)
        ax.set_title(macro_name)

        # axis cosmetics (Nature-like)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(direction="out", length=3, width=0.8)
        ax.set_xlim(x_min, x_max)

        # faint reference at 0
        ax.axvline(0.0, linewidth=0.8, alpha=0.35)
        ax.axhline(0.0, linewidth=0.0)  # keep clean; no grid by default
        ax.grid(False)

        # plot per country
        for c in sorted(subm["country_input"].unique()):
            g = subm[subm["country_input"] == c].copy()
            if g.empty:
                continue

            # if add_band=True, keep multiple sentences per country; else single sentence already filtered above
            if add_band:
                # use all sentence_id remaining for this country+macro+category
                # build aligned interpolation grid
                series = []
                x_common = None
                for sid in sorted(g["sentence_id"].unique()):
                    gs = g[g["sentence_id"] == sid].sort_values("macro_value_std")
                    x = gs["macro_value_std"].values.astype(float)
                    y = gs["prob"].values.astype(float)
                    if interpolate_points and interpolate_points > 2:
                        xi, yi = _interp(x, y, n=interpolate_points)
                    else:
                        xi, yi = x, y
                    if x_common is None:
                        x_common = xi
                    else:
                        # align by interpolation onto x_common
                        yi = np.interp(x_common, xi, yi)
                    series.append(yi)
                y_stack = np.vstack(series) if len(series) else None
                if y_stack is None:
                    continue
                mu, lo, hi = _band_stats(y_stack)

                ax.fill_between(x_common, lo, hi, alpha=0.15, linewidth=0.0)
                ax.plot(
                    x_common, mu,
                    linewidth=line_width,
                    label=label_map.get(c, c)
                )
                if show_markers:
                    # overlay sparse markers at original grid points (first sentence) for readability
                    gs0 = g[g["sentence_id"] == sorted(g["sentence_id"].unique())[0]].sort_values("macro_value_std")
                    ax.plot(gs0["macro_value_std"].values, gs0["prob"].values,
                            linestyle="None", marker="o", markersize=marker_size, alpha=0.8)
            else:
                # single curve
                g = g.sort_values("macro_value_std")
                x = g["macro_value_std"].values.astype(float)
                y = g["prob"].values.astype(float)

                if interpolate_points and interpolate_points > 2:
                    xi, yi = _interp(x, y, n=interpolate_points)
                else:
                    xi, yi = x, y

                ax.plot(
                    xi, yi,
                    linewidth=line_width,
                    label=label_map.get(c, c)
                )
                if show_markers:
                    ax.plot(
                        x, y,
                        linestyle="None",
                        marker="o",
                        markersize=marker_size,
                        alpha=0.85
                    )

        ax.set_xlabel("Standardized value (σ)")
        ax.set_ylabel("Predicted probability")

    # remove extra axes if any
    for j in range(len(macro_cols_order), nrows * ncols):
        fig.delaxes(axes[j])

    # compact global legend
    handles, labels = axes[0].get_legend_handles_labels()
    # de-duplicate legend entries
    seen = set()
    h2, l2 = [], []
    for h, l in zip(handles, labels):
        if l in seen:
            continue
        seen.add(l)
        h2.append(h)
        l2.append(l)

    fig.legend(
        h2, l2,
        loc="upper center",
        ncol=legend_ncol,
        frameon=False,
        handlelength=2.4,
        columnspacing=1.2
    )

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(output_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)

    return output_png


In [ ]:
# 最小修改：在函数签名里加一个参数（country_colors），然后在 ax.plot(...) / fill_between(...) 里用它。

# 1) 在函数定义参数里加这一行：
# country_colors: dict = None,

# 2) 在 plot 循环里（for c in ...）加两行：
# color = None if (country_colors is None) else country_colors.get(c, None)
# 然后把 ax.plot(... ) / ax.fill_between(... ) 都加上 color=color

# 具体改动（直接复制替换对应位置）：

def plot_nhb_comparison_from_csv(
    curves_csv: str,
    output_png: str,
    summary_csv: str = None,
    sentence_length_filter: str = "long",
    sentence_id_filter: list = None,
    choose_sentence_per_country: str = "max_baseline_prob",
    plot_mode: str = "top1_by_baseline",
    fixed_category: str = None,
    macro_cols_order: list = None,
    interpolate_points: int = 200,
    add_band: bool = False,
    band_stat: str = "std",
    nrows: int = 2,
    ncols: int = 3,
    figsize=(8.5, 5.6),
    dpi: int = 300,
    font_family: str = "Arial",
    base_fontsize: int = 9,
    line_width: float = 1.6,
    marker_size: float = 3.0,
    show_markers: bool = True,
    legend_ncol: int = 2,
    country_colors: dict = None,     # <- (1) 新增
):
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    df = pd.read_csv(curves_csv)
    df = df[df["mode"] == "one_at_a_time"].copy()

    if sentence_length_filter is not None:
        df = df[df["sentence_length"] == sentence_length_filter].copy()

    if sentence_id_filter is not None:
        df = df[df["sentence_id"].isin(sentence_id_filter)].copy()

    if macro_cols_order is None:
        macro_cols_order = list(df["macro"].dropna().unique())
    macro_cols_order = [m for m in macro_cols_order if m != "__all__"]
    macro_cols_order = macro_cols_order[: (nrows * ncols)]

    if sentence_id_filter is None and not add_band:
        chosen = []
        for c in df["country_input"].unique():
            sub = df[df["country_input"] == c].copy()
            if sub.empty:
                continue
            if choose_sentence_per_country == "max_baseline_prob":
                sid = (
                    sub.groupby("sentence_id")["baseline_prob"]
                    .max()
                    .sort_values(ascending=False)
                    .index[0]
                )
            else:
                sid = sub["sentence_id"].iloc[0]
            chosen.append((c, sid))
        keep = pd.DataFrame(chosen, columns=["country_input", "sentence_id"])
        df = df.merge(keep, on=["country_input", "sentence_id"], how="inner")

    if plot_mode == "fixed_category":
        if fixed_category is None:
            raise ValueError("plot_mode='fixed_category' requires fixed_category")
        df_plot = df[df["category"] == fixed_category].copy()
        label_map = {c: f"{c} ({fixed_category})" for c in df_plot["country_input"].unique()}
    else:
        pick = (
            df.groupby(["country_input", "macro", "category"], as_index=False)["baseline_prob"]
            .max()
            .sort_values(["country_input", "macro", "baseline_prob"], ascending=[True, True, False])
            .groupby(["country_input", "macro"], as_index=False)
            .head(1)
        )
        df_plot = df.merge(
            pick[["country_input", "macro", "category"]],
            on=["country_input", "macro", "category"],
            how="inner"
        )
        tmp = (
            df_plot.groupby(["country_input", "category"], as_index=False)["baseline_prob"]
            .max()
            .sort_values(["country_input", "baseline_prob"], ascending=[True, False])
            .groupby("country_input", as_index=False)
            .head(1)
        )
        cat_by_country = dict(zip(tmp["country_input"], tmp["category"]))
        label_map = {c: f"{c} ({cat_by_country.get(c,'')})" for c in df_plot["country_input"].unique()}

    plt.rcParams.update({
        "font.family": font_family,
        "font.size": base_fontsize,
        "axes.titlesize": base_fontsize + 1,
        "axes.labelsize": base_fontsize,
        "legend.fontsize": base_fontsize - 1,
        "xtick.labelsize": base_fontsize - 1,
        "ytick.labelsize": base_fontsize - 1,
        "axes.linewidth": 0.8,
        "figure.dpi": dpi,
        "savefig.dpi": dpi,
    })

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=True, sharey=False)
    axes = np.array(axes).reshape(-1)

    x_min = df_plot["macro_value_std"].min()
    x_max = df_plot["macro_value_std"].max()

    def _interp(x, y, n=200):
        xi = np.linspace(float(np.min(x)), float(np.max(x)), int(n))
        yi = np.interp(xi, x, y)
        return xi, yi

    def _band_stats(y_stack):
        if band_stat == "sem":
            mu = np.nanmean(y_stack, axis=0)
            sd = np.nanstd(y_stack, axis=0, ddof=1)
            n = np.sum(np.isfinite(y_stack), axis=0)
            se = np.divide(sd, np.sqrt(np.maximum(n, 1)), out=np.zeros_like(sd), where=n > 0)
            return mu, mu - se, mu + se
        if band_stat == "iqr":
            mu = np.nanmedian(y_stack, axis=0)
            q1 = np.nanpercentile(y_stack, 25, axis=0)
            q3 = np.nanpercentile(y_stack, 75, axis=0)
            return mu, q1, q3
        mu = np.nanmean(y_stack, axis=0)
        sd = np.nanstd(y_stack, axis=0, ddof=1)
        return mu, mu - sd, mu + sd

    for i, macro_name in enumerate(macro_cols_order):
        ax = axes[i]
        subm = df_plot[df_plot["macro"] == macro_name].copy()
        ax.set_title(macro_name)

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(direction="out", length=3, width=0.8)
        ax.set_xlim(x_min, x_max)
        ax.axvline(0.0, linewidth=0.8, alpha=0.35)
        ax.grid(False)

        for c in sorted(subm["country_input"].unique()):
            g = subm[subm["country_input"] == c].copy()
            if g.empty:
                continue

            color = None if (country_colors is None) else country_colors.get(c, None)  # <- (2) 新增

            if add_band:
                series = []
                x_common = None
                for sid in sorted(g["sentence_id"].unique()):
                    gs = g[g["sentence_id"] == sid].sort_values("macro_value_std")
                    x = gs["macro_value_std"].values.astype(float)
                    y = gs["prob"].values.astype(float)
                    if interpolate_points and interpolate_points > 2:
                        xi, yi = _interp(x, y, n=interpolate_points)
                    else:
                        xi, yi = x, y
                    if x_common is None:
                        x_common = xi
                    else:
                        yi = np.interp(x_common, xi, yi)
                    series.append(yi)

                y_stack = np.vstack(series) if len(series) else None
                if y_stack is None:
                    continue
                mu, lo, hi = _band_stats(y_stack)

                ax.fill_between(x_common, lo, hi, alpha=0.15, linewidth=0.0, color=color)  # <- color
                ax.plot(x_common, mu, linewidth=line_width, label=label_map.get(c, c), color=color)  # <- color
                if show_markers:
                    gs0 = g[g["sentence_id"] == sorted(g["sentence_id"].unique())[0]].sort_values("macro_value_std")
                    ax.plot(gs0["macro_value_std"].values, gs0["prob"].values,
                            linestyle="None", marker="o", markersize=marker_size, alpha=0.8, color=color)  # <- color
            else:
                g = g.sort_values("macro_value_std")
                x = g["macro_value_std"].values.astype(float)
                y = g["prob"].values.astype(float)

                if interpolate_points and interpolate_points > 2:
                    xi, yi = _interp(x, y, n=interpolate_points)
                else:
                    xi, yi = x, y

                ax.plot(xi, yi, linewidth=line_width, label=label_map.get(c, c), color=color)  # <- color
                if show_markers:
                    ax.plot(x, y, linestyle="None", marker="o", markersize=marker_size, alpha=0.85, color=color)  # <- color

        ax.set_xlabel("Standardized value (σ)")
        ax.set_ylabel("Predicted probability")

    for j in range(len(macro_cols_order), nrows * ncols):
        fig.delaxes(axes[j])

    handles, labels = axes[0].get_legend_handles_labels()
    seen = set()
    h2, l2 = [], []
    for h, l in zip(handles, labels):
        if l in seen:
            continue
        seen.add(l)
        h2.append(h)
        l2.append(l)

    fig.legend(h2, l2, loc="upper center", ncol=legend_ncol, frameon=False,
               handlelength=2.4, columnspacing=1.2)

    plt.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(output_png, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return output_png


In [ ]:
curves_csv, summary_csv = build_structural_sensitivity_csv_v3(
    model=model,
    meta=meta,
    year_val=2021,
    output_dir="./outputs",
    output_prefix="6ctry_6macros_6sents",
    base_macro_mode="zeros",
    store_mode="topk",
    topk=10,
    macro_grid_spec={
        # 需要的话逐个变量定制扫描范围/点数
        "unemployment_value": (-2, 2, 21),
        "inflation_value": (-2, 2, 21),
        "battle_value": (0, 2, 11),
        "debt_value": (-2, 2, 21),
        "trade_value": (-2, 2, 21),
        "gdp_value": (-2, 2, 21),
    },
)

In [ ]:
# 1) One-line usage (recommended default)
country_colors = {
    "United States":  "#0072B2",  # blue
    "United Kingdom": "#D55E00",  # vermillion
    "Japan":          "#009E73",  # bluish green
    "South Korea":    "#CC79A7",  # reddish purple
    "Germany":        "#56B4E9",  # sky blue
    "Finland":        "#E69F00",  # orange
}

file_types = ['S1', 'S2', 'S3', 'L1', 'L2', 'L3', ]
for file_type in file_types:
    png_path = plot_nhb_comparison_from_csv(
        curves_csv="./outputs/6ctry_6macros_6sents_curves.csv",
        output_png=f"./outputs/{file_type}_compare.pdf",
        sentence_id_filter=[f'US_{file_type}',f'JP_{file_type}',f'UK_{file_type}',f'DE_{file_type}',f'FI_{file_type}',f'KR_{file_type}'],
        sentence_length_filter="short",
        plot_mode="top1_by_baseline",
        interpolate_points=200,
        add_band=False,
        country_colors=country_colors
    )
    print(png_path)
